# 03 — Grid Search Mlp / 03 Grid Search Mlp

**Chapter 15 — File 3 of 5 / 第15章 — 第3个文件（共5个）**

---

## Summary / 总结

This script demonstrates **grid search mlps for monthly airline passengers dataset**.

本脚本演示 **grid search mlps for monthly airline passengers dataset**。

---
## Step 1 — grid search mlps for monthly airline passengers dataset

In [ ]:
from math import sqrt
from numpy import array
from numpy import mean
from pandas import DataFrame
from pandas import concat
from pandas import read_csv
from sklearn.metrics import mean_squared_error
from keras.models import Sequential
from keras.layers import Dense

---
## Step 2 — split a univariate dataset into train/test sets

In [ ]:
def train_test_split(data, n_test):
	return data[:-n_test], data[-n_test:]

---
## Step 3 — transform list into supervised learning format

In [ ]:
def series_to_supervised(data, n_in, n_out=1):
	df = DataFrame(data)
	cols = list()

---
## Step 4 — input sequence (t-n, ... t-1)

In [ ]:
for i in range(n_in, 0, -1):
		cols.append(df.shift(i))

---
## Step 5 — forecast sequence (t, t+1, ... t+n)

In [ ]:
for i in range(0, n_out):
		cols.append(df.shift(-i))

---
## Step 6 — put it all together

In [ ]:
agg = concat(cols, axis=1)

---
## Step 7 — drop rows with NaN values

In [ ]:
agg.dropna(inplace=True)
	return agg.values

---
## Step 8 — root mean squared error or rmse

In [ ]:
def measure_rmse(actual, predicted):
	return sqrt(mean_squared_error(actual, predicted))

---
## Step 9 — difference dataset

In [ ]:
def difference(data, order):
	return [data[i] - data[i - order] for i in range(order, len(data))]

---
## Step 10 — fit a model

In [ ]:
def model_fit(train, config):

---
## Step 11 — unpack config

In [ ]:
n_input, n_nodes, n_epochs, n_batch, n_diff = config

---
## Step 12 — prepare data

In [ ]:
if n_diff > 0:
		train = difference(train, n_diff)

---
## Step 13 — transform series into supervised format

In [ ]:
data = series_to_supervised(train, n_in=n_input)

---
## Step 14 — separate inputs and outputs

In [ ]:
train_x, train_y = data[:, :-1], data[:, -1]

---
## Step 15 — define model

In [ ]:
model = Sequential()
	model.add(Dense(n_nodes, activation='relu', input_dim=n_input))
	model.add(Dense(1))
	model.compile(loss='mse', optimizer='adam')

---
## Step 16 — fit model

In [ ]:
model.fit(train_x, train_y, epochs=n_epochs, batch_size=n_batch, verbose=0)
	return model

---
## Step 17 — forecast with the fit model

In [ ]:
def model_predict(model, history, config):

---
## Step 18 — unpack config

In [ ]:
n_input, _, _, _, n_diff = config

---
## Step 19 — prepare data

In [ ]:
correction = 0.0
	if n_diff > 0:
		correction = history[-n_diff]
		history = difference(history, n_diff)

---
## Step 20 — shape input for model

In [ ]:
x_input = array(history[-n_input:]).reshape((1, n_input))

---
## Step 21 — make forecast

In [ ]:
yhat = model.predict(x_input, verbose=0)

---
## Step 22 — correct forecast if it was differenced

In [ ]:
return correction + yhat[0]

---
## Step 23 — walk-forward validation for univariate data

In [ ]:
def walk_forward_validation(data, n_test, cfg):
	predictions = list()

---
## Step 24 — split dataset

In [ ]:
train, test = train_test_split(data, n_test)

---
## Step 25 — fit model

In [ ]:
model = model_fit(train, cfg)

---
## Step 26 — seed history with training dataset

In [ ]:
history = [x for x in train]

---
## Step 27 — step over each time-step in the test set

In [ ]:
for i in range(len(test)):

---
## Step 28 — fit model and make forecast for history

In [ ]:
yhat = model_predict(model, history, cfg)

---
## Step 29 — store forecast in list of predictions

In [ ]:
predictions.append(yhat)

---
## Step 30 — add actual observation to history for the next loop

In [ ]:
history.append(test[i])

---
## Step 31 — estimate prediction error

In [ ]:
error = measure_rmse(test, predictions)
	print(' > %.3f' % error)
	return error

---
## Step 32 — score a model, return None on failure

In [ ]:
def repeat_evaluate(data, config, n_test, n_repeats=10):

---
## Step 33 — convert config to a key

In [ ]:
key = str(config)

---
## Step 34 — fit and evaluate the model n times

In [ ]:
scores = [walk_forward_validation(data, n_test, config) for _ in range(n_repeats)]

---
## Step 35 — summarize score

In [ ]:
result = mean(scores)
	print('> Model[%s] %.3f' % (key, result))
	return (key, result)

---
## Step 36 — grid search configs

In [ ]:
def grid_search(data, cfg_list, n_test):

---
## Step 37 — evaluate configs

In [ ]:
scores = scores = [repeat_evaluate(data, cfg, n_test) for cfg in cfg_list]

---
## Step 38 — sort configs by error, asc

In [ ]:
scores.sort(key=lambda tup: tup[1])
	return scores

---
## Step 39 — create a list of configs to try

In [ ]:
def model_configs():

---
## Step 40 — define scope of configs

In [ ]:
n_input = [12]
	n_nodes = [50, 100]
	n_epochs = [100]
	n_batch = [1, 150]
	n_diff = [0, 12]

---
## Step 41 — create configs

In [ ]:
configs = list()
	for i in n_input:
		for j in n_nodes:
			for k in n_epochs:
				for l in n_batch:
					for m in n_diff:
						cfg = [i, j, k, l, m]
						configs.append(cfg)
	print('Total configs: %d' % len(configs))
	return configs

---
## Step 42 — define dataset

In [ ]:
series = read_csv('monthly-airline-passengers.csv', header=0, index_col=0)
data = series.values

---
## Step 43 — data split

In [ ]:
n_test = 12

---
## Step 44 — model configs

In [ ]:
cfg_list = model_configs()

---
## Step 45 — grid search

In [ ]:
scores = grid_search(data, cfg_list, n_test)
print('done')

---
## Step 46 — list top 3 configs

In [ ]:
for cfg, error in scores[:3]:
	print(cfg, error)

---
## Learning Notes / 学习笔记

- **概念**: grid search mlps for monthly airline passengers dataset 是机器学习中的常用技术。  
  *grid search mlps for monthly airline passengers dataset is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Grid Search Mlp / 03 Grid Search Mlp
# Complete Code / 完整代码
# ===============================

# grid search mlps for monthly airline passengers dataset
from math import sqrt
from numpy import array
from numpy import mean
from pandas import DataFrame
from pandas import concat
from pandas import read_csv
from sklearn.metrics import mean_squared_error
from keras.models import Sequential
from keras.layers import Dense

# split a univariate dataset into train/test sets
def train_test_split(data, n_test):
	return data[:-n_test], data[-n_test:]

# transform list into supervised learning format
def series_to_supervised(data, n_in, n_out=1):
	df = DataFrame(data)
	cols = list()
	# input sequence (t-n, ... t-1)
	for i in range(n_in, 0, -1):
		cols.append(df.shift(i))
	# forecast sequence (t, t+1, ... t+n)
	for i in range(0, n_out):
		cols.append(df.shift(-i))
	# put it all together
	agg = concat(cols, axis=1)
	# drop rows with NaN values
	agg.dropna(inplace=True)
	return agg.values

# root mean squared error or rmse
def measure_rmse(actual, predicted):
	return sqrt(mean_squared_error(actual, predicted))

# difference dataset
def difference(data, order):
	return [data[i] - data[i - order] for i in range(order, len(data))]

# fit a model
def model_fit(train, config):
	# unpack config
	n_input, n_nodes, n_epochs, n_batch, n_diff = config
	# prepare data
	if n_diff > 0:
		train = difference(train, n_diff)
	# transform series into supervised format
	data = series_to_supervised(train, n_in=n_input)
	# separate inputs and outputs
	train_x, train_y = data[:, :-1], data[:, -1]
	# define model
	model = Sequential()
	model.add(Dense(n_nodes, activation='relu', input_dim=n_input))
	model.add(Dense(1))
	model.compile(loss='mse', optimizer='adam')
	# fit model
	model.fit(train_x, train_y, epochs=n_epochs, batch_size=n_batch, verbose=0)
	return model

# forecast with the fit model
def model_predict(model, history, config):
	# unpack config
	n_input, _, _, _, n_diff = config
	# prepare data
	correction = 0.0
	if n_diff > 0:
		correction = history[-n_diff]
		history = difference(history, n_diff)
	# shape input for model
	x_input = array(history[-n_input:]).reshape((1, n_input))
	# make forecast
	yhat = model.predict(x_input, verbose=0)
	# correct forecast if it was differenced
	return correction + yhat[0]

# walk-forward validation for univariate data
def walk_forward_validation(data, n_test, cfg):
	predictions = list()
	# split dataset
	train, test = train_test_split(data, n_test)
	# fit model
	model = model_fit(train, cfg)
	# seed history with training dataset
	history = [x for x in train]
	# step over each time-step in the test set
	for i in range(len(test)):
		# fit model and make forecast for history
		yhat = model_predict(model, history, cfg)
		# store forecast in list of predictions
		predictions.append(yhat)
		# add actual observation to history for the next loop
		history.append(test[i])
	# estimate prediction error
	error = measure_rmse(test, predictions)
	print(' > %.3f' % error)
	return error

# score a model, return None on failure
def repeat_evaluate(data, config, n_test, n_repeats=10):
	# convert config to a key
	key = str(config)
	# fit and evaluate the model n times
	scores = [walk_forward_validation(data, n_test, config) for _ in range(n_repeats)]
	# summarize score
	result = mean(scores)
	print('> Model[%s] %.3f' % (key, result))
	return (key, result)

# grid search configs
def grid_search(data, cfg_list, n_test):
	# evaluate configs
	scores = scores = [repeat_evaluate(data, cfg, n_test) for cfg in cfg_list]
	# sort configs by error, asc
	scores.sort(key=lambda tup: tup[1])
	return scores

# create a list of configs to try
def model_configs():
	# define scope of configs
	n_input = [12]
	n_nodes = [50, 100]
	n_epochs = [100]
	n_batch = [1, 150]
	n_diff = [0, 12]
	# create configs
	configs = list()
	for i in n_input:
		for j in n_nodes:
			for k in n_epochs:
				for l in n_batch:
					for m in n_diff:
						cfg = [i, j, k, l, m]
						configs.append(cfg)
	print('Total configs: %d' % len(configs))
	return configs

# define dataset
series = read_csv('monthly-airline-passengers.csv', header=0, index_col=0)
data = series.values
# data split
n_test = 12
# model configs
cfg_list = model_configs()
# grid search
scores = grid_search(data, cfg_list, n_test)
print('done')
# list top 3 configs
for cfg, error in scores[:3]:
	print(cfg, error)

---

➡️ **Next / 下一步**: File 4 of 5